# Vision-Language Models (VLM): Make an LLM Understand Images

> **Background**: GPT-4V, Claude, and Gemini can all "talk about images". How do they do it?
>
> Goal for this part: **understand the core VLM idea: how to turn an image into a token sequence that an LLM can consume.**

One-sentence core idea:
**image -> split into patches -> each patch becomes a vector -> treat them as special "visual tokens" -> concatenate with text tokens -> feed to the LLM.**


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

torch.manual_seed(42)

### 1. The core problem: images and text are totally different

```
Text: "a cat sits on a mat"
  -> Tokenizer -> [12, 45, 78, 3, 90, 23]   (1D token sequence)

Image: a 224x224 cat photo
  -> ??? -> [?, ?, ?, ...]                  (how to make tokens?)
```

LLMs only know token sequences. So the VLM task is:

**convert an image into a sequence of tokens, so the LLM can "read" it like text.**

A common 3-step recipe:
1. split the image into patches
2. turn each patch into a vector (patch embedding)
3. treat those vectors as tokens and concatenate with text tokens


### 2. Step 1: patchify the image

Like turning a big photo into a jigsaw puzzle. Each patch is a "visual word".

```
Original image: 224 x 224
Patch size: 16 x 16
-> 14 x 14 = 196 patches

+--+--+--+--+
|  |  |  |  |  each patch is 16x16 pixels
+--+--+--+--+
|  |🐱|  |  |  cat face is in one patch
+--+--+--+--+
|  |  |  |  |
+--+--+--+--+
```

Why 16x16? That was the choice in the original ViT paper, and it became a standard.


In [2]:
#  image：3 (RGB) × 224 × 224
IMG_SIZE = 224
PATCH_SIZE = 16

#  image( )
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print(f" image : {fake_image.shape}")
print(f" : [3 , 224, 224]")

# split intopatch
# unfold ： 
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE).unfold(2, PATCH_SIZE, PATCH_SIZE)
print(f"\n : {patches.shape}")
print(f" : [3 , 14 patch, 14 patch, 16 , 16 ]")

#  [num_patches, patch_dim]
num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE  # 3×16×16 = 768
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print(f"\n : {patches_flat.shape}")
print(f" : [{num_patches} patch, patch {patch_dim} ]")
print(f"\n : text token 1 -> vision token {patch_dim} ")
print(f" ： {patch_dim} text token ")

 image : torch.Size([3, 224, 224])
 : [3 , 224, 224]

 : torch.Size([3, 14, 14, 16, 16])
 : [3 , 14 patch, 14 patch, 16 , 16 ]

 : torch.Size([196, 768])
 : [196 patch, patch 768 ]

 : text token 1 -> vision token 768 
 ： 768 text token 


### 3. Step 2: patch embedding (turn each patch into a vector)

Each patch has `3*16*16 = 768` raw pixel values. But the LLM embedding dimension might be 512/768/...

Use a linear projection from 768 -> d_model so visual tokens and text tokens live in the same vector space.

```
patch (768) -> Linear(768, d_model) -> visual token (d_model)
text  "cat"  -> embedding lookup     -> text token   (d_model)

Same dimension -> can be concatenated.
```


In [3]:
class PatchEmbedding(nn.Module):
    """ imagesplit intopatch, patch vector ViT 「Tokenizer」—— image """
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        
        #  ： , 「 + 」
        # kernel_size=patch_size, stride=patch_size -> split intopatch
        self.proj = nn.Conv2d(in_channels, d_model, 
                              kernel_size=patch_size, stride=patch_size)
    
    def forward(self, x):
        """ : x shape = [batch, 3, 224, 224] : shape = [batch, num_patches, d_model] """
        x = self.proj(x)          # [batch, d_model, 14, 14]
        x = x.flatten(2)           # [batch, d_model, 196]
        x = x.transpose(1, 2)      # [batch, 196, d_model]
        return x

#  
patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)  # batch=2 image
visual_tokens = patch_emb(dummy_img)

print(f" image: {dummy_img.shape}")
print(f"vision token: {visual_tokens.shape}")
print(f" : [2 image, 196 visiontoken, 512 ]")
print(f"\n vision token text token ！ .")

 image: torch.Size([2, 3, 224, 224])
vision token: torch.Size([2, 196, 512])
 : [2 image, 196 visiontoken, 512 ]

 vision token text token ！ .


### 4. Step 3: visual tokens + text tokens -> feed the LLM together

This is the key operation: concatenate both modalities into one token sequence.

```
Input sequence:
[<img>] [v1] [v2] ... [v196] [</img>] [please] [describe] [this] [image]
  ^       ^    ^           ^     ^        ^       ^        ^      ^
 special  patch tokens                special    text tokens
```

The LLM just sees a long vector sequence. It does not intrinsically care which came from image vs text.


In [4]:
#  VLM concatenate
d_model = 512
text_vocab_size = 1000

#  text tokenizer ID
text_ids = torch.tensor([[5, 12, 78, 3, 90]])  # " "
text_emb = nn.Embedding(text_vocab_size, d_model)
text_vecs = text_emb(text_ids)  # [1, 5, 512]

# vision token( PatchEmbedding )
visual_vecs = torch.randn(1, 196, d_model)  #  196 vision token

#  ！
combined = torch.cat([visual_vecs, text_vecs], dim=1)

print(f"vision token: {visual_vecs.shape}")
print(f"text token: {text_vecs.shape}")
print(f"concatenate : {combined.shape}")
print(f"\n : [1 , 196+5=201 token, 512 ]")
print(f"LLM 201 token , 196 image")

vision token: torch.Size([1, 196, 512])
text token: torch.Size([1, 5, 512])
concatenate : torch.Size([1, 201, 512])

 : [1 , 196+5=201 token, 512 ]
LLM 201 token , 196 image


### 5. A complete VLM architecture sketch

```
                 Image (224x224x3)
                      |
                      v
           +----------------------+   196 patches -> vectors
           |   Patch Embedding    |   (often Conv2d with kernel=16,stride=16)
           +----------+-----------+
                      | [196, d_model]
                      v
           +----------------------+   optional: a few Transformer layers
           |   Vision Encoder     |   to let visual tokens interact
           +----------+-----------+
                      |
      +---------------+---------------------------+
      |                                           |
      v                                           v
 [visual tokens 1..196]                    [text tokens ...]
      |                                           |
      +------------------ concatenate ------------+
                      |
                      v
              +---------------+
              |  LLM Backbone |
              | (Transformer) |
              +-------+-------+
                      |
                      v
            "This is a cat on a mat"
```

Key insight: **you often do not need to change the LLM itself**.
Just add an "image -> tokens" front-end.


### 6. Three common VLM architectures (detailed)

In industry there are three main ways to inject image information into an LLM.
The key difference is:

- **Where** the visual features enter the LLM
- Whether the **LLM architecture needs to change**

Here is the punchline first:

```text
                  Modify LLM?   Visual tokens in sequence?   Representative models
Visual tokens         no               196+                 LLaVA / LLaVA-1.5 / GPT-4V
Cross-attention       yes               0                   Flamingo / OpenFlamingo
Q-Former              no               32                   BLIP-2 / InstructBLIP
```

#### 6.1 Visual-token approach (LLaVA-style) - the most common

We already used this approach earlier:

`image -> patches -> vectors -> concatenate before text tokens -> feed to the LLM`.

The best part: **the LLM does not need to change at all**.
It simply treats visual tokens and text tokens as the same kind of token sequence.

#### 6.2 Cross-attention approach (Flamingo-style) - the most elegant

**Core idea**: do not mix visual features into the text token sequence.
Instead, keep visual features "on the side".
At each layer the LLM uses **cross-attention** to *query* the image.

```text
Text token stream (normal flow):
  Layer 1: Self-Attn -> Cross-Attn -> FFN -> hidden_1
  Layer 2: Self-Attn -> Cross-Attn -> FFN -> hidden_2
  Layer 3: Self-Attn -> Cross-Attn -> FFN -> hidden_3
  ...
                    ^              ^              ^
                     \             |             /
                      Visual features [N, d]  (persistent, do not move)
```

**Self-Attention vs Cross-Attention: the real difference**

```text
Self-Attention:
  Q, K, V all come from the same sequence
  -> tokens "look at" each other inside one modality
  -> complexity O(seq_len^2)

Cross-Attention:
  Q comes from text hidden states (current LLM state)
  K, V come from visual features (image encoder outputs)
  -> text retrieves information from images
  -> complexity O(text_len * num_visual_features)
```

**Cross-Attention formula** (same as self-attention, except the source of K/V):

$$\text{CrossAttn}(X_{text}, X_{vis}) = \text{softmax}\left(\frac{Q_{text} K_{vis}^T}{\sqrt{d_k}}\right) V_{vis}$$

Where:
- $Q_{text} = X_{text} W^Q$ (from text hidden states)
- $K_{vis} = X_{vis} W^K$, $V_{vis} = X_{vis} W^V$ (from visual features)

**Flamingo's key trick: Tanh gating**

A naive cross-attention block would add the cross-attention output directly.
Flamingo uses a **tanh gate** instead:

$$X_{out} = X_{in} + \tanh(\alpha) \cdot \text{CrossAttn}(X_{in}, X_{vis})$$

$\alpha$ is a **learnable scalar**, initialized to 0.

- At the start of training: `tanh(0) = 0`, cross-attention is effectively off
- As training progresses: $\alpha$ grows, cross-attention gradually turns on

This makes the visual capability "grow" on top of the existing language ability,
instead of destroying it from day 1.

**Why use cross-attention?**

1. Visual features do not consume the text sequence length, so you can handle more images.
2. The same image features can be reused across layers efficiently.
3. Gating protects the base LLM during early training.


# A full PyTorch implementation of Cross-Attention
# Compare Self-Attention and Cross-Attention and see the key difference

```python
class CrossAttention(nn.Module):
    """
    Cross-Attention layer: Q comes from text, K/V come from images.

    The only difference from Self-Attention is the source of K and V:
    - Self-Attn: Q, K, V come from the same sequence
    - Cross-Attn: K, V come from "external" visual features
    """

    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)   # Q: text side
        self.k_proj = nn.Linear(d_model, d_model)   # K: image side
        self.v_proj = nn.Linear(d_model, d_model)   # V: image side
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, text_hidden, image_features):
        """
        Args:
            text_hidden:    [B, T, D]  LLM hidden states (Q)
            image_features: [B, N, D]  vision encoder outputs (K, V)

        Returns:
            [B, T, D] text states updated with image information
        """
        B, T, D = text_hidden.shape
        N = image_features.shape[1]

        # Q from text, K/V from image
        Q = self.q_proj(text_hidden)      # [B, T, D]
        K = self.k_proj(image_features)   # [B, N, D]
        V = self.v_proj(image_features)   # [B, N, D]

        # Multi-head reshape
        Q = Q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, T, d]
        K = K.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]
        V = V.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)  # [B, h, N, d]

        # Scaled dot-product attention
        # Q @ K^T: [B, h, T, d] @ [B, h, d, N] -> [B, h, T, N]
        # Meaning: how much the t-th text token attends to the n-th visual feature
        attn_weights = (Q @ K.transpose(-2, -1)) * self.scale           # [B, h, T, N]
        attn_weights = F.softmax(attn_weights, dim=-1)

        # Weighted aggregation: [B, h, T, N] @ [B, h, N, d] -> [B, h, T, d]
        # Meaning: pull image information into text representations
        out = attn_weights @ V                                          # [B, h, T, d]
        out = out.transpose(1, 2).reshape(B, T, D)                       # [B, T, D]
        return self.out_proj(out)


class FlamingoGatedCrossAttnBlock(nn.Module):
    """
    Flamingo-style Transformer block:
    Self-Attention -> Gated Cross-Attention -> FFN

    Key trick: tanh(alpha) gating. alpha starts at 0 so the visual path turns on gradually.
    """

    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        # Self-Attention (standard)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        # Cross-Attention (vision injection)
        self.cross_attn = CrossAttention(d_model, n_heads)
        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        # Layer norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        # The gating parameter - Flamingo's key idea
        self.alpha = nn.Parameter(torch.zeros(1))  # initialized to 0

    def forward(self, x, image_features, causal_mask=None):
        """
        x:              [B, T, D] text hidden states
        image_features: [B, N, D] visual features
        """
        # 1) Self-attention: text-text interaction
        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)

        # 2) Gated cross-attention: text looks at image
        #    tanh(0) ~= 0 -> image path is suppressed at the start
        gate = torch.tanh(self.alpha)
        x = x + gate * self.cross_attn(x, image_features)
        x = self.norm2(x)

        # 3) FFN
        x = x + self.ffn(x)
        x = self.norm3(x)
        return x


# Baseline comparison: a pure LLM block (no cross-attn) vs Flamingo block (with cross-attn)
class LLaVABlock(nn.Module):
    """A standard LLM block: only self-attention. Visual tokens are concatenated into the sequence."""

    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, causal_mask=None):
        x = x + self.self_attn(x, x, x, attn_mask=causal_mask)[0]
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x


# --- Quick test & comparison ---
print("=== Cross-Attention vs Visual Tokens ===\n")

d_model, n_heads = 512, 8

# Mock inputs
batch = 1
text_len = 50
num_vis = 196
text_hidden = torch.randn(batch, text_len, d_model)
image_feat = torch.randn(batch, num_vis, d_model)

# LLaVA-style: concatenate -> longer sequence
combined = torch.cat([image_feat, text_hidden], dim=1)  # [1, 196+50, 512]
llava_block = LLaVABlock(d_model, n_heads)
llava_out = llava_block(combined)
print("LLaVA (Visual Tokens):")
print(f"  input: [{batch}, {text_len + num_vis}, {d_model}]")
print(f"  self-attention sees: {text_len + num_vis} tokens")
print(f"  FLOPs ~ (196+50)^2 = {246**2:,}")

# Flamingo-style: cross-attention -> text length unchanged
flamingo_block = FlamingoGatedCrossAttnBlock(d_model, n_heads)
with torch.no_grad():
    flamingo_out = flamingo_block(text_hidden, image_feat)
print("\nFlamingo (Cross-Attention):")
print(f"  input: [{batch}, {text_len}, {d_model}]")
print(f"  self-attention sees: {text_len} tokens")
print(f"  cross-attention: Q({text_len}) @ K({num_vis})")
print(f"  FLOPs ~ 50^2 + 50*196 = {50**2 + 50*196:,}")
print(f"  gate alpha = {flamingo_block.alpha.item():.4f} -> tanh(alpha) = {torch.tanh(flamingo_block.alpha).item():.4f}")

print("\nConclusion:")
print("  LLaVA: longer sequence -> more FLOPs, but no LLM architecture change -> faster to build")
print("  Flamingo: sequence length unchanged -> fewer FLOPs, but you must modify the LLM -> higher engineering cost")
print("  Tanh gate (alpha=0): cross-attn is suppressed at the start -> protects language ability")
```


### 7. Why do images consume so many tokens?

A 224x224 image, split into 16x16 patches, becomes **196 visual tokens**.

Compare that to text: a short sentence might be only ~10 tokens.
So in token budget, an image is often "much longer" than a sentence.

This is a major reason VLM inference is slower:

- A plain LLM: input 50 tokens -> fast
- A VLM: input 50 + 196 = 246 tokens -> much slower (attention is **O(n^2)**)

**Common optimization directions**:

- Use larger patches (32x32 -> only 49 tokens)
- Use Q-Former compression (196 -> 32 tokens)
- Use dynamic resolution (small images use fewer tokens, large images use more)


# Get an intuition: how many visual tokens for different patch sizes

```python
for patch_size in [8, 14, 16, 32]:
    num_patches = (224 // patch_size) ** 2
    print(f"patch_size={patch_size:2d} -> {num_patches:4d} visual tokens")

print("\nBigger patches -> fewer tokens -> faster inference")
print("But patches that are too big -> each token must carry too much detail -> you lose information")
print("16x16 is a common sweet spot in practice")
```


### 8. Training and freezing strategy for VLMs

A VLM is rarely trained "all at once".
The central training question is: among the three components

- Vision encoder
- Projector (connector)
- LLM

**which do you train first, and which do you freeze?**

#### 8.1 Why do we need freezing?

```text
Vision encoder (ViT):  ~300M params   already pre-trained on massive image-text data
LLM:                  ~7B params     already pre-trained on massive text data
Projector:            ~8M params     starts from scratch -> must be trained
```

If you try to train everything from scratch:

- GPU memory will not fit (full gradients + optimizer states for 7B + 300M)
- you will destroy the existing vision and language abilities (catastrophic forgetting)
- the dataset size is usually not enough (a few million image-text pairs is far less than pretraining)

So you almost always do **selective freezing**.

#### 8.2 The classic two-stage LLaVA strategy (most important)

```text
Stage 1: Alignment pre-training (feature alignment)

         Vision encoder -> Projector -> LLM
            freeze          train       freeze

Goal:   teach the projector to translate vision features into vectors the LLM can use
Data:   image-text pairs (image + one-sentence caption), ~558K
Loss:   standard next-token prediction (LLM outputs the caption)
Key:    because the LLM is frozen, you only train the projector (~8M params) -> fast and cheap

Stage 2: Instruction tuning

         Vision encoder -> Projector -> LLM
            freeze          train       train

Goal:   learn to answer diverse questions about images (not only captioning)
Data:   images + multi-turn dialogues / QA / reasoning, ~665K
Loss:   compute loss only on text; visual token positions are ignored
Key:    the vision encoder is still frozen - its feature extraction is already good enough
```

#### 8.3 Why is the vision encoder almost always frozen?

1. CLIP/SigLIP-style encoders are already strong feature extractors.
2. The VLM mainly needs to learn **how to use** visual features, not how to extract them.
3. Unfreezing often causes catastrophic forgetting because VLM data is much smaller than vision pretraining.
4. Memory: ViT-L is ~300M parameters. Unfreezing it makes training much heavier.

Exception: some work unfreezes the last few vision layers in stage 2, but that is research-grade tuning.

#### 8.4 Why freeze the LLM in stage 1, then unfreeze in stage 2?

```text
Stage 1: freeze the LLM
  - the projector outputs "weird vectors" at the start; training the LLM on them can hurt language ability
  - first make the projector produce vectors that look like text embeddings
  - only ~8M projector params -> quick alignment

Stage 2: unfreeze the LLM
  - now the projector is aligned, so the LLM can safely learn to use visual information
  - this is where true "visual QA" capability grows
```

#### 8.5 Alternative: use LoRA to tune the LLM

Full finetuning a 7B LLM can require ~60GB memory. LoRA reduces it a lot:

```text
Full finetune:
  need weights + grads + optimizer states
  7B bf16 ~ 14GB (weights) + 14GB (grads) + 28GB (optim) ~ 56GB

LoRA finetune:
  freeze base weights, train low-rank adapters
  7B bf16 ~ 14GB (weights) + ~0.5GB (LoRA) + ~1GB (optim) ~ 16GB
  trainable params: ~50M (< 1%), often close to full finetune quality
```

```python
# Pseudocode: LLaVA-style freezing logic
def configure_training(vision_encoder, projector, llm, stage=1):
    """Decide what to train."""
    # Vision encoder: always frozen
    for p in vision_encoder.parameters():
        p.requires_grad = False

    # Projector: always trained
    for p in projector.parameters():
        p.requires_grad = True

    # LLM: depends on the stage
    if stage == 1:
        for p in llm.parameters():
            p.requires_grad = False
        print("Stage 1 (alignment): train Projector only")
    elif stage == 2:
        for p in llm.parameters():
            p.requires_grad = True
        print("Stage 2 (instruction tuning): train Projector + LLM (full or LoRA)")
    else:
        raise ValueError(f"Unknown stage: {stage}")

print("=== LLaVA freezing strategy (conceptual) ===\n")
print("Stage 1: alignment")
print("  trainable: Projector (~8M)")
print("  frozen: Vision encoder (~300M) + LLM (~7B)")
print("  data: image -> caption")
print()
print("Stage 2: instruction tuning")
print("  trainable: Projector + LLM")
print("  frozen: Vision encoder")
print("  data: image + multi-turn dialogues")
```

#### 8.6 Freezing strategy comparison (quick table)

| Model | Vision encoder | Projector | LLM | Notes |
|------|:---:|:---:|:---:|------|
| LLaVA 1.5 | frozen | train | stage1 frozen, stage2 train | classic two-stage |
| LLaVA 1.6 | frozen | train | stage1 frozen, stage2 train | adds AnyRes |
| Flamingo | frozen | train (cross-attn) | frozen | LLM is never trained; relies on tanh gate |
| BLIP-2 | frozen | train (Q-Former) | frozen | two-stage Q-Former training |
| mPLUG-Owl | frozen | train | train (LoRA) | LoRA in stage 2 |
| CogVLM | frozen | train | train (partial) | per-layer visual experts |

**A useful rule of thumb**:

- Vision encoder: almost always frozen
- Projector: almost always trained
- LLM: freeze in stage 1, train in stage 2 (full or LoRA)
- Flamingo is an exception: keep the LLM frozen and learn vision via cross-attention + gating

#### 8.7 A decision tree (when choosing a strategy)

```text
If you want to train a VLM:

- Plenty of compute (e.g. 8x A100): use the standard LLaVA two-stage recipe
- Limited compute (1-2x A100): use BLIP-2 style (keep LLM frozen)
- Protect base LLM behavior at all costs: Flamingo style (LLM frozen + gated cross-attn)
- Quick experiments: use LoRA everywhere you can
```


### 7.5 Engineering details for vision injection: projector, positional encoding, special tokens

We said "concatenate visual tokens before text tokens".
But in practice, several engineering details determine whether it works well.

#### 7.5.1 Projector (connector): the core of dimension alignment

The vision encoder output dimension is often different from the LLM hidden size.
So you need a projector to map one space into the other.

**LLaVA's evolution**:

```text
LLaVA v1:   Linear(d_vis -> d_llm)
LLaVA 1.5:  MLP(d_vis -> d_llm) with 2 layers + GELU
LLaVA 1.6:  MLP + AnyRes (dynamic tiling)
```

**Why upgrade from Linear to MLP?**

Vision features and text features are not just "different in dimension".
Their distributions and information density differ.
A single Linear layer is mostly rotation + scaling.
A 2-layer MLP adds nonlinearity and can learn a richer mapping.

```python
# LLaVA 1.5 style projector
class LLaVAProjector(nn.Module):
    """Map visual features into the LLM embedding space."""

    def __init__(self, vis_dim=1024, llm_dim=4096, hidden_multiplier=2):
        super().__init__()
        hidden_dim = vis_dim * hidden_multiplier
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, llm_dim),
        )

    def forward(self, visual_features):
        """visual_features: [B, N, vis_dim] -> [B, N, llm_dim]"""
        return self.mlp(visual_features)
```

#### 7.5.2 Special tokens: mark the boundaries of images

When visual tokens are concatenated into the same sequence, how does the LLM know
which tokens belong to an image?
Some models use special tokens to mark boundaries:

```text
<image_start> [vis_1] [vis_2] ... [vis_196] <image_end> <bos> Describe the image <eos>
     ^                                         ^
  image starts                              image ends
```

Different models choose different conventions:

| Model | Approach | Notes |
|------|------|------|
| LLaVA | no special tokens | fixed 196 visual tokens at the front; the LLM learns the convention |
| Qwen-VL | `<img>...</img>` | explicit wrappers, supports multiple images |
| InternVL | `<IMG_CONTEXT>` | one context token per image |
| Gemini-style | no explicit tokens | image/text interleave; use segment IDs |

**For multi-image inputs**, special tokens become more important:

```text
<img> [vis...(img1)] </img> What is in the first image?
<img> [vis...(img2)] </img> How does the second differ from the first?
```

#### 7.5.3 Positional encoding: what positions should visual tokens use?

A subtle but important question: after concatenation, how do we index positions?

```text
Option A: one shared 1D position space (LLaVA-style)
  positions: [0..195, 196, 197, ...]
  tokens:    [vis_0..vis_195, text_0, text_1, ...]

Option B: separate 2D positions for vision + 1D positions for text
  more faithful to image geometry, but harder to integrate into a 1D LLM

Option C: learned visual positional embeddings
  a separate learned table for vision positions, not shared with text
```

LLaVA uses Option A because it is simple and works surprisingly well.

Why positions matter:

- Image patches have spatial structure.
- A 1D ordering can put non-neighbor patches next to each other in index space.
- 2D positions (or 2D-RoPE) preserve geometry, but most LLMs assume 1D RoPE.

#### 7.5.4 Multi-resolution and dynamic tiling

A 224x224 image with 16x16 patches is 196 tokens.
But what if a user uploads a 4K image?

```text
Option A: forced resize
  4K -> resize to 224x224
  downside: you lose text details and small objects

Option B: AnyRes (LLaVA 1.6)
  tile the image into multiple 224x224 crops
  each crop -> 196 tokens
  if there are N crops, tokens ~ (1 + N) * 196

Option C: dynamic patch size
  smaller patches -> more tokens, potentially better detail (but slower)
```

AnyRes-style concatenation often looks like:

```text
[thumbnail(224x224)] [crop1(224x224)] [crop2(224x224)] ... [cropN(224x224)] [text]
    196 tokens          196 tokens        196 tokens              196 tokens
```

#### 7.5.5 Which vision layer should you use?

```python
vision_output = clip_model.vision_model(pixel_values)

# Option 1: final layer
features = vision_output.last_hidden_state  # [B, 196, 1024]

# Option 2: penultimate layer (LLaVA 1.5)
features = vision_output.hidden_states[-2]  # [B, 196, 1024]

# Option 3: fuse multiple layers
features = vision_output.hidden_states[-4:].mean(dim=0)
```

LLaVA 1.5 reports that the penultimate layer can work better than the final layer:
final-layer features can be too "high-level" and lose low-level details that matter.


### 8. Training: usually two stages

VLMs are typically trained in two stages:

Stage 1: alignment / pretraining
- goal: align visual tokens with text space
- data: lots of image-caption pairs
- often train only the projector, freeze vision encoder + LLM

Stage 2: instruction tuning
- goal: answer questions about images
- data: image + question/answer pairs
- train projector + (some of) LLM; often keep vision encoder frozen

```
Stage 1: image -> projector -> frozen LLM -> caption
Stage 2: image + question -> LLM -> answer
```


### 9. A minimal PyTorch VLM

Now we implement a tiny VLM that runs.


---

## VLM Summary (Checklist)

### What you learned

1. [ok] **Core idea**: image -> patches -> vectors -> treat as tokens -> feed into the LLM
2. [ok] **Patch embedding** is the image "tokenizer". A Conv2d can do patchify + projection in one shot.
3. [ok] **Visual-token concatenation**: after dimension alignment, concatenate visual tokens with text tokens; the LLM can stay unchanged.
4. [ok] **Cross-attention (detailed)**:
   - Q comes from text hidden states, K/V come from visual features
   - Flamingo uses **tanh gating**: `tanh(alpha)` with `alpha=0` at init to protect language ability
   - vs visual tokens: cross-attn does not consume sequence length, but requires architecture changes
5. [ok] **Engineering details**:
   - projector choices: Linear (LLaVA v1) -> MLP+GELU (LLaVA 1.5) -> Q-Former (BLIP-2)
   - special tokens: `<img>...</img>` to mark image boundaries (Qwen-VL / InternVL style)
   - positional encoding: shared 1D order vs separate 2D vs learned visual positions
   - multi-resolution: forced resize vs AnyRes tiling vs dynamic patch size
   - vision features often use the **penultimate** vision layer (more low-level detail)
6. [ok] **Training and freezing**:
   - vision encoder: almost always frozen
   - projector: almost always trained
   - LLM: stage 1 frozen (alignment), stage 2 trained (instruction tuning)
   - LoRA can replace full finetuning: < 1% trainable params, much less memory
   - Flamingo is an exception: LLM stays frozen; cross-attn + gate learns vision
7. [ok] **Three architectures**: Visual tokens (LLaVA) / Cross-attention (Flamingo) / Q-Former (BLIP-2)
8. [ok] **Images are expensive in token budget** (196+ tokens), which is a main reason VLM inference is slower

### Quick glossary

| Concept | One-line explanation |
|------|-----------|
| Patch embedding | `Conv2d(16x16, stride=16)`: 224x224 image -> 196 tokens |
| Projector | translator: vision feature space -> LLM embedding space |
| Cross-attention | text queries visual features via Q(text), K/V(image) |
| Tanh gating | `tanh(alpha)` gate; with `alpha=0`, cross-attn is off at the start |
| Freezing strategy | vision frozen, projector trained, LLM trained mainly in stage 2 |
| AnyRes | tile a large image into many 224x224 crops, encode each, then concatenate |
| Q-Former | 32 learnable queries compress 196 features into ~32 tokens |
| LoRA for VLM | freeze base weights, train low-rank adapters to save memory |
| Special tokens | wrappers like `<img>...</img>` to support multi-image interleaving |

### Architecture choice (rule of thumb)

```text
- Fast to build: LLaVA (visual tokens)
- Multi-image interleaving: Flamingo (cross-attention)
- Inference speed priority: BLIP-2 (Q-Former compression)
- Limited GPU memory: LLaVA + LoRA
```

**One-sentence VLM summary**: give the LLM "eyes". The vision encoder sees, the projector translates, and the LLM understands and answers.


In [5]:
class MiniVLM(nn.Module):
    """ Visual Token VLM：PatchEmbedding + text embedding + TransformerEncoder"""
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(img_size=224, patch_size=16, in_channels=3, d_model=d_model)
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        x = torch.cat([visual_tokens, text_tokens], dim=1)
        x = self.layers(x)
        x = self.norm(x)
        return self.lm_head(x)

#  MiniVLM
vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

#  
dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))  # 10 text token

logits = vlm(dummy_images, dummy_text)

print(f" : image [{dummy_images.shape}] + text [{dummy_text.shape}]")
print(f" : {logits.shape}")
print(f" : [1 , 196+10=206 , 100 ]")
print(f"\n : {sum(p.numel() for p in vlm.parameters()):,}")

 : image [torch.Size([1, 3, 224, 224])] + text [torch.Size([1, 10])]
 : torch.Size([1, 206, 100])
 : [1 , 196+10=206 , 100 ]

 : 520,932


---

## VLM Summary (One line)

VLMs work by turning images into token-like vectors and feeding them into an LLM along with text.
